In [1]:
import sys, os
sys.path.append(os.path.abspath('../..'))
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

base_folder = "/data/big_rim/rsync_dcc_sum/Oct3V1" #"/data/big_rim/rsync_dcc_sum/24summ" #"/data/big_rim/rsync_dcc_sum/25Apri_social" #"/data/big_rim/rsync_dcc_sum/Oct3V1" #"/hpc/group/tdunn/Bryan_Rigs/BigOpenField/24summ"  # Replace with your base folder
# save_path = os.path.join(base_folder, 'paret')
failed_paths_file = '/data/big_rim/rsync_dcc_sum/Oct3V1/sync_failed_brws.txt'  # File containing failed paths


force_rescan_rec_files = [
    # ('2023-10-01', '001'),
    # ('2023-10-02', '002'),
    # Add more as needed
]
rescan_threshold_days = 0.0001 # 7 days, but guess if i mess up i can just change it to automatically rescan all, smile... #0.1

log_folder_to_parquet_sep(base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
                            force_rescan_rec_files=force_rescan_rec_files,
                            rescan_threshold_days=rescan_threshold_days)


all_df = read_all_parquet_files(base_folder)

Log for trashmini_240916v1r1_miceleash_test_19_15 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_01_23/trashmini_240916v1r1_miceleash_test_19_15/folder_log.parquet
Log for 20241217v1l23re1_p20241217v1l23BE0 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1_p20241217v1l23BE0/folder_log.parquet
Log for 20241217v1l23re1 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1/folder_log.parquet
Log for #calib_before saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/#calib_before/folder_log.parquet
Log for 24Anshu_f_paint_2mice_2 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/24Anshu_f_paint_2mice_2/folder_log.parquet
Log for #Anshu_f_bleach_2mice saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/#Anshu_f_bleach_2mice/folder_log.parquet
Log for 24Anshu_f_paint_2mice saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/24Anshu_f_paint_2mice/folder_log.parquet
Log for 24Ansh

In [2]:
import pyarrow.compute as pc
from functools import reduce


table = all_df #combined_df
# Filter mir_generate_param == 0 and sync != 3
conditions = [
   # pc.equal(table['mir_generate_param'], '0'),
   pc.equal(table['sync'], '1'),
   # pc.not_equal(table['sync'], '3'),
   # pc.equal(table['com'], '1'),
   # # pc.equal(table['com_vis'], '1'),
   # # pc.equal(table['v1'], '1'),
   # pc.equal(table['dannce'], '1'),
   #  
   # pc.equal(table['test'], '1'),
   # pc.equal(table['dannce_vis'], '1'),
   pc.equal(table['social'], '1'),  
   # pc.equal(table['mini_6cam_map'], '1'),
   # pc.equal(table['date_folder'], '2025_05_16'),
   pc.equal(table['mini_rec_sync_com'], '0'),
   # pc.equal(table['mini_rec_sync'], '1'),
   #mini_rec_sync mini_rec_sync_com
   # mini_6cam_map
]

filter_mask = reduce(pc.and_, conditions)


# Apply the filter and print the results
social_beh = table.filter(filter_mask)

# Print each row of the filtered table
print(social_beh.to_pandas())  # This will display the filtered data in a familiar pandas-like format


   mir_generate_param sync mini_6cam_map dropf_handle com com_vis social  \
0                   1    1             1            0   1       1      1   
1                   1    1             0            0   1       1      1   
2                   1    1             0            0   1       1      1   
3                   1    1             0            0   1       1      1   
4                   1    1             0            0   1       0      1   
5                   1    1             0            0   1       1      1   
6                   1    1             0            0   1       1      1   
7                   1    1             0            0   1       1      1   
8                   1    1             0            0   1       0      1   
9                   1    1             0            0   1       0      1   
10                  1    1             1            0   1       1      1   
11                  1    1             1            0   1       1      1   
12          

In [3]:
# for cases if i did not produce, or get rid of the produced files. ie so that i would hav eto find out the miniscope qualities.
# import pyarrow.compute as pc
# from functools import reduce
# import pandas as pd
# import os

# # Step 1: Get all social recordings with sync
# conditions = [
#     pc.equal(table['sync'], '1'),
#     pc.equal(table['social'], '0'),
# ]
# filter_mask = reduce(pc.and_, conditions)
# filtered_table = table.filter(filter_mask)

# # Convert to pandas for easier manipulation
# df = filtered_table.to_pandas()

# # Step 2: FIXED function - use 'quality' column not 'condition'
# def get_mini_quality(rec_path):
#     """
#     Check if miniscope recording has good quality by reading folder_log.parquet.
#     Returns: 'good', 'good_vein', 'bad', or None
#     """
#     sync_file = os.path.join(rec_path, 'sync_to_mini_path.txt')
    
#     if not os.path.exists(sync_file):
#         return None
    
#     try:
#         # Read the mini path from sync_to_mini_path.txt
#         with open(sync_file, 'r') as f:
#             mini_path = f.read().strip()
        
#         # Look for folder_log.parquet in the mini folder
#         parquet_file = os.path.join(mini_path, 'folder_log.parquet')
        
#         if not os.path.exists(parquet_file):
#             return None
        
#         # Read the parquet file
#         quality_df = pd.read_parquet(parquet_file)
        
#         # Check if 'quality' column exists (NOT 'condition'!)
#         if 'quality' in quality_df.columns and len(quality_df) > 0:
#             return quality_df['quality'].iloc[0]
        
#         return None
#     except Exception as e:
#         print(f"Error reading quality for {rec_path}: {e}")
#         return None

# # Step 3: Add quality column
# print("Reading mini quality from parquet files...")
# df['mini_quality'] = df['rec_path'].apply(get_mini_quality)

# # Step 4: Create final filter
# df['include_mini'] = (
#     (df['mini_6cam_map'] == '0') |  # Behavior only
#     ((df['mini_6cam_map'] == '1') & (df['mini_quality'].isin(['good', 'good_vein'])))
# )

# # Step 5: Show results
# good_mini_recordings = df[df['include_mini'] == True]

# print("\nRecordings with good mini or behavior-only:")
# print(good_mini_recordings['rec_path'].tolist())

# # Show the breakdown
# has_good_mini = df[(df['mini_6cam_map'] == '1') & (df['mini_quality'].isin(['good', 'good_vein']))]
# behavior_only = df[df['mini_6cam_map'] == '0']
# bad_mini = df[(df['mini_6cam_map'] == '1') & (~df['mini_quality'].isin(['good', 'good_vein']))]

# print(f"\nWith good mini: {len(has_good_mini)}")
# print(f"Behavior only: {len(behavior_only)}")
# print(f"Bad mini (excluded): {len(bad_mini)}")

# # Show quality distribution for mapped recordings
# print("\nQuality distribution for mini_6cam_map==1:")
# print(df[df['mini_6cam_map'] == '1']['mini_quality'].value_counts())

In [6]:
# for socials
from export_plannings import seed_pairs_csv, autofill_short_ids, build_plan_from_pairs
import pyarrow as pa
from export_profiles import load_profiles

profiles = load_profiles("profiles.yaml")
profile = profiles["default"]
# ============================================================================
# NEW WORKFLOW: One CSV per filtered table
# ============================================================================

# Define your table name
table_name = "social_beh"  # ← Change this for each different table

# Auto-generate CSV name from table name
csv_file = f"./{table_name}_pairs.csv"

print(f"📋 Working with table: {table_name}")
print(f"📁 CSV file: {csv_file}")

# 1. Create/update CSV
seed_pairs_csv(
    filtered_table=social_beh,  # ← Your filtered table variable
    out_csv=csv_file,
    prev_csv=csv_file  # Reuse same file to preserve existing rows
)

# 2. Manually edit: fill in animal1_key, animal2_key
print("\n⏸️  PAUSE HERE!")
print(f"   Open {csv_file} and fill in animal names for new rows")
input("   Press Enter when done...")

# # 3. Auto-assign IDs (uses registry - IDs stay consistent!)
autofill_short_ids(csv_file)

# # 4. Build export plan
plan = build_plan_from_pairs(csv_file, profile)

print(f"\n✅ Plan ready with {len(plan)} sessions")

/tmp/ipykernel_175067/913204672.py:22: FutureWarning: promote has been superseded by promote_options='default'.
  seed_pairs_csv(


📋 Working with table: social_beh
📁 CSV file: ./social_beh_pairs.csv

⏸️  PAUSE HERE!
   Open ./social_beh_pairs.csv and fill in animal names for new rows

✅ Plan ready with 25 sessions


In [7]:
plan

[{'dest_rel_path': 'social/MC7+UNK1_S1/',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_07/2social_mini_20241015pmcr2_single_AO_13_24',
  'id': 'MC7+UNK1',
  'session_index': 1,
  'path_globs': ['videos/**',
   'df_*_label3d_dannce.mat',
   'folder_log.parquet',
   'MIR_Aligned/frame_mapping.json']},
 {'dest_rel_path': 'social/VC12+VC13_S1/',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1_p20241217v1l23BE0',
  'id': 'VC12+VC13',
  'session_index': 1,
  'path_globs': ['videos/**',
   'df_*_label3d_dannce.mat',
   'folder_log.parquet',
   'MIR_Aligned/frame_mapping.json']},
 {'dest_rel_path': 'social/MC10+UNK1_S1/',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_29/2social_mini_0605pmc_single_18_00',
  'id': 'MC10+UNK1',
  'session_index': 1,
  'path_globs': ['videos/**',
   'df_*_label3d_dannce.mat',
   'folder_log.parquet',
   'MIR_Aligned/frame_mapping.json']},
 {'dest_rel_path': 'social/MC10+UNK1_S2/',
  'sour

In [10]:
from export_profiles import export_with_rsync_preserve_tree
from export_profiles import load_profiles

profiles = load_profiles("profiles.yaml")
profile = profiles["default"]
plan = build_plan_from_pairs(csv_file, profile)

for it in plan:
    it["is_social"] = 1  # all rows here are social

folder_nameeee = "oct3v1_social_bev"

export_with_rsync_preserve_tree(
    plan,
    f"/data/big_rim/sorted_mir_dataset/{folder_nameeee}",  # export_root (2nd positional)
    profile,                                                  # profile (3rd positional)
    dry_run=False,
    copy_miniscope=False,
    mode="staged",
    profile_label="default",
    log_parquet_path=f"export_logs/export_log_{folder_nameeee}.parquet",
    mini_csv_paths=[  # ← ADD THIS
        "/home/lq53/mir_repos/BBOP/random_tests/25may_mini/mini_nc_mapping_250618.csv",
        "/home/lq53/mir_repos/BBOP/random_tests/25may_mini/mini_nc_mapping_250725_sum_0618andprev.csv"
    ],
    mini_includes=["*.avi", "timeStamps.csv"] #"*.avi", "*.png", #png file is used for validation only. it was validated so we can exclude now...
)

[CLEANUP DEBUG] Pattern: **/metadata/*.png
[CLEANUP DEBUG] Full pattern: /data/big_rim/sorted_mir_dataset/oct3v1_social_bev/**/metadata/*.png
[CLEANUP DEBUG] Found 0 matches

[CLEANUP DEBUG] Total to delete: 0
[TIDY] Moving folder_log.parquet -> metadata/
[TIDY] Renaming df_intriUpdated_df_synced_2024_11_07_2social_mini_20241015pmcr2_single_AO_13_24_calib_before_label3d_dannce.mat -> label3d_dannce.mat
[TIDY] Moving folder_log.parquet -> metadata/
[TIDY] Renaming df_intriUpdated_df_synced_2025_05_21_20241217v1l23re1_p20241217v1l23BE0_calib_before_label3d_dannce.mat -> label3d_dannce.mat
[TIDY] Moving folder_log.parquet -> metadata/
[TIDY] Renaming df_intriUpdated_df_synced_2024_10_29_2social_mini_0605pmc_single_18_00_calib_before_label3d_dannce.mat -> label3d_dannce.mat
[TIDY] Moving folder_log.parquet -> metadata/
[TIDY] Renaming df_intriUpdated_df_synced_2024_10_29_2social_mini_0605pmc_single_18_25_calib_before_label3d_dannce.mat -> label3d_dannce.mat
[TIDY] Moving folder_log.parquet